# Initiation à l'API Berserk

L'objectif de ce notebook est de comprendre comment utiliser l'API Berserk.

## 1. Imports de libraries

In [1]:
from dotenv import load_dotenv

import os
import sys
sys.path.append(os.path.abspath(".."))

from pprint import pprint

import berserk
import pandas as pd 

## 2. Chargement du token

In [2]:
load_dotenv("../.env")

token = os.getenv("LICHESS_API_TOKEN")

if not token:
    raise ValueError("LICHESS_API_TOKEN introuvable dans le fichier .env")

print("Token chargé avec succès")

Token chargé avec succès


## 3. Connexion à l'API Berserk

In [3]:
session = berserk.TokenSession(token)
client = berserk.Client(session=session)

print("Client Berserk créé avec succès")

Client Berserk créé avec succès


## 4. Affichage des informations de mon profil Lichess

In [4]:
account = client.account.get()
pprint(account)

{'blocking': False,
 'count': {'all': 1,
           'bookmark': 0,
           'draw': 0,
           'import': 0,
           'loss': 0,
           'me': 0,
           'playing': 0,
           'rated': 1,
           'win': 1},
 'createdAt': datetime.datetime(2026, 3, 17, 18, 3, 37, 330000, tzinfo=datetime.timezone.utc),
 'followable': True,
 'following': False,
 'id': 'raydennnnnnnnn',
 'perfs': {'blitz': {'games': 1,
                     'prog': 0,
                     'prov': True,
                     'rating': 1674,
                     'rd': 297},
           'bullet': {'games': 0,
                      'prog': 0,
                      'prov': True,
                      'rating': 1500,
                      'rd': 500},
           'classical': {'games': 0,
                         'prog': 0,
                         'prov': True,
                         'rating': 1500,
                         'rd': 500},
           'correspondence': {'games': 0,
                              'prog'

## 5. Importation des parties d'un joueur

Dans cet exemple, il est importé les parties du joueur Chesstam. Un maximim de 5 parties sont importées. Seules des parties jouées en mode rapide sont considérées.

In [5]:
username = "Chesstam"

games = client.games.export_by_player(
    username,
    max=150,
    perf_type="rapid",
    opening=True,
)

games = list(games)

print(f"{len(games)} parties récupérées")

150 parties récupérées


## 6. Inspection d'une partie

Cette première commande permet d'extraire toutes les informations d'une partie. L'API a converti le format PGN en JSON.

In [6]:
pprint(games[0])

{'clock': {'increment': 0, 'initial': 600, 'totalTime': 600},
 'createdAt': datetime.datetime(2026, 3, 20, 16, 9, 2, 740000, tzinfo=datetime.timezone.utc),
 'id': '15RbtDUM',
 'lastMoveAt': datetime.datetime(2026, 3, 20, 16, 18, 49, 421000, tzinfo=datetime.timezone.utc),
 'moves': 'd4 d5 Nc3 e6 a3 c6 Bf4 Bd6 Nh3 Bxf4 Nxf4 Nd7 e3 Ngf6 h3 O-O Bd3 e5 '
          'dxe5 Nxe5 g4 Nxd3+ Qxd3 Be6 O-O-O Qd7 Qe2 Rad8 g5 Ne8 h4 Bg4 f3 Bf5 '
          'h5 f6 g6 h6 e4 Be6 exd5 Bxd5 Nfxd5 cxd5 Nxd5 Nd6 Ne7+ Kh8 Nd5 Rfe8 '
          'Qe3 Rxe3 Nxe3 Qe6 Rhe1 Qh3 f4 Qxh5 f5 Qh2 Rxd6 Rxd6 Ng2 Rd8 Ne3 Qf4 '
          'Kb1 Kg8 Nc4',
 'opening': {'eco': 'D00',
             'name': "Queen's Pawn Game: Chigorin Variation",
             'ply': 4},
 'perf': 'rapid',
 'players': {'black': {'rating': 1420,
                       'ratingDiff': 5,
                       'user': {'id': 'pizza2s3ananasom9',
                                'name': 'Pizza2s3ananasom9'}},
             'white': {'rating': 1368,
          

Cette seconde commande permet d'extraire les différents champs d'une partie.

In [7]:
print(sorted(games[0].keys()))

for key in sorted(games[0].keys()):
    print(key)

['clock', 'createdAt', 'id', 'lastMoveAt', 'moves', 'opening', 'perf', 'players', 'rated', 'source', 'speed', 'status', 'variant', 'winner']
clock
createdAt
id
lastMoveAt
moves
opening
perf
players
rated
source
speed
status
variant
winner


Les prochaines commandes permettent d'afficher les sous-champs des champs "players" et "opening".

In [8]:
for key in sorted(games[0]["players"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"]["user"].keys()):
    print(key)

print("\n")

black
white


rating
ratingDiff
user


id
name




In [9]:
for key in sorted(games[0]["opening"].keys()):
    print(key)

eco
name
ply


# 7. Applatissement d'une partie

La commande suivante éxécute une fonction permettant de convertir una partie brute en une structure plate.

In [10]:
from src.ingestion.flatten_games import flatten_game

flat = flatten_game(games[0])
flat

{'game_id': '15RbtDUM',
 'datetime_utc': datetime.datetime(2026, 3, 20, 16, 9, 2, 740000, tzinfo=datetime.timezone.utc),
 'perf': 'rapid',
 'speed': 'rapid',
 'rated': True,
 'source': 'pool',
 'status': 'resign',
 'winner': 'black',
 'white_id': 'chesstam',
 'white_name': 'Chesstam',
 'black_id': 'pizza2s3ananasom9',
 'black_name': 'Pizza2s3ananasom9',
 'white_rating_before': 1368,
 'black_rating_before': 1420,
 'white_rating_diff': -5,
 'black_rating_diff': 5,
 'white_rating_after': 1363,
 'black_rating_after': 1425,
 'opening_eco': 'D00',
 'opening_name': "Queen's Pawn Game: Chigorin Variation",
 'clock_initial': 600,
 'clock_increment': 0,
 'has_increment': 0,
 'moves': 'd4 d5 Nc3 e6 a3 c6 Bf4 Bd6 Nh3 Bxf4 Nxf4 Nd7 e3 Ngf6 h3 O-O Bd3 e5 dxe5 Nxe5 g4 Nxd3+ Qxd3 Be6 O-O-O Qd7 Qe2 Rad8 g5 Ne8 h4 Bg4 f3 Bf5 h5 f6 g6 h6 e4 Be6 exd5 Bxd5 Nfxd5 cxd5 Nxd5 Nd6 Ne7+ Kh8 Nd5 Rfe8 Qe3 Rxe3 Nxe3 Qe6 Rhe1 Qh3 f4 Qxh5 f5 Qh2 Rxd6 Rxd6 Ng2 Rd8 Ne3 Qf4 Kb1 Kg8 Nc4',
 'ply_count': 69}

# 8. Conversion en une partie du point de vue du joueur

La commande suivante fait appel à une fonction permettant de transformer une partie aplatie en 2 rangées (1 par joueur).

In [11]:
from src.ingestion.player_view import game_to_player_rows

flat = flatten_game(games[1])
rows = game_to_player_rows(flat)

rows

[{'game_id': 'WYwK05zv',
  'datetime_utc': datetime.datetime(2026, 3, 20, 11, 37, 12, 271000, tzinfo=datetime.timezone.utc),
  'weekday': 4,
  'player_id': 'mlimok',
  'player_name': 'mlimok',
  'opponent_id': 'chesstam',
  'opponent_name': 'Chesstam',
  'color_player': 'white',
  'result_player': 1.0,
  'elo_player_before': 1388,
  'elo_player_after': 1394,
  'elo_diff_player': 6,
  'termination_type': 'resign',
  'has_increment': 0,
  'opening_family': 'Scandinavian Defense',
  'opening_eco': 'B01',
  'opening_name': 'Scandinavian Defense: Mieses-Kotroc Variation',
  'ply_count': 77,
  'perf': 'rapid',
  'speed': 'rapid',
  'rated': True,
  'source': 'pool'},
 {'game_id': 'WYwK05zv',
  'datetime_utc': datetime.datetime(2026, 3, 20, 11, 37, 12, 271000, tzinfo=datetime.timezone.utc),
  'weekday': 4,
  'player_id': 'chesstam',
  'player_name': 'Chesstam',
  'opponent_id': 'mlimok',
  'opponent_name': 'mlimok',
  'color_player': 'black',
  'result_player': 0.0,
  'elo_player_before': 137

# 9. Table complète des parties du joueur

La commande suivante permet de convertir un ensemble de parties brutes en un ensemble de parties aplaties en 2 rangées (format de la section précédente).

In [12]:
from src.ingestion.build_player_games import build_player_games

df_player_games = build_player_games(games)

df_player_games = df_player_games[df_player_games["player_id"] == "chesstam"].reset_index(drop=True)

print(len(games))

print(len(df_player_games))

print(df_player_games.columns)

df_player_games.head(150)



150
150
Index(['game_id', 'datetime_utc', 'weekday', 'player_id', 'player_name',
       'opponent_id', 'opponent_name', 'color_player', 'result_player',
       'elo_player_before', 'elo_player_after', 'elo_diff_player',
       'termination_type', 'has_increment', 'opening_family', 'opening_eco',
       'opening_name', 'ply_count', 'perf', 'speed', 'rated', 'source'],
      dtype='str')


,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,termination_type,has_increment,opening_family,opening_eco,opening_name,ply_count,perf,speed,rated,source
0,15RbtDUM,2026-03-20 16:09:02.740000+00:00,4,chesstam,Chesstam,pizza2s3ananasom9,Pizza2s3ananasom9,white,0.0,1368,...,resign,0,Queen's Pawn Game,D00,Queen's Pawn Game: Chigorin Variation,69,rapid,rapid,True,pool
1,WYwK05zv,2026-03-20 11:37:12.271000+00:00,4,chesstam,Chesstam,mlimok,mlimok,black,0.0,1373,...,resign,0,Scandinavian Defense,B01,Scandinavian Defense: Mieses-Kotroc Variation,77,rapid,rapid,True,pool
2,SgMdaJWB,2026-03-19 19:27:52.047000+00:00,3,chesstam,Chesstam,grgab2,Grgab2,white,0.0,1378,...,resign,0,"Rapport-Jobava System, with e6",D01,"Rapport-Jobava System, with e6",61,rapid,rapid,True,pool
3,P9He4fsU,2026-03-19 19:14:55.157000+00:00,3,chesstam,Chesstam,mustafa_t_20,mustafa_t_20,white,1.0,1373,...,resign,0,"Rapport-Jobava System, with e6",D01,"Rapport-Jobava System, with e6",109,rapid,rapid,True,pool
4,QCCqlAgv,2026-03-19 19:07:35.777000+00:00,3,chesstam,Chesstam,tramputin,tramputin,black,0.5,1372,...,draw,0,Scandinavian Defense,B01,Scandinavian Defense: Mieses-Kotroc Variation,113,rapid,rapid,True,pool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,6vzH2flj,2026-03-06 21:58:16.384000+00:00,4,chesstam,Chesstam,alienuno,alienuno,white,1.0,1390,...,resign,0,Queen's Pawn Game,D00,"Queen's Pawn Game: Chigorin Variation, Irish G...",55,rapid,rapid,True,pool
146,OtI44dvX,2026-03-06 21:44:57.365000+00:00,4,chesstam,Chesstam,mohamed_mohamed_1111,Mohamed_Mohamed_1111,black,1.0,1385,...,timeout,0,Scandinavian Defense,B01,Scandinavian Defense,126,rapid,rapid,True,pool
147,ixcGj2wv,2026-03-06 21:29:47.688000+00:00,4,chesstam,Chesstam,tonysaika,tonySaika,white,0.5,1385,...,draw,0,Rapport-Jobava System,D01,Rapport-Jobava System,93,rapid,rapid,True,pool
148,5b3HoutL,2026-03-06 21:23:21.590000+00:00,4,chesstam,Chesstam,zinkael,zinkael,black,1.0,1379,...,resign,0,English Opening,A10,English Opening: Anglo-Scandinavian Defense,52,rapid,rapid,True,pool


# 10. Implémentation des features partie "entre-parties"

In [13]:
from src.features.temporal_features import add_basic_temporal_features

df_player_games_v2 = add_basic_temporal_features(df_player_games)

df_player_games_v2[["player_id", "index", "delay_previous_game", "streak_before", "streak_after"]].head(150)


,player_id,index,delay_previous_game,streak_before,streak_after
149,chesstam,0,NaN,NaN,0
148,chesstam,1,814.234,0.0,1
147,chesstam,2,386.098,1.0,0
146,chesstam,3,909.677,0.0,1
145,chesstam,4,799.019,1.0,2
...,...,...,...,...,...
4,chesstam,145,271.139,1.0,0
3,chesstam,146,439.380,0.0,1
2,chesstam,147,776.890,1.0,-1
1,chesstam,148,58160.224,-1.0,-2


# 11 Création des séries temporelles

## 11.1 Série temporelle session

L'expression ci-dessous fait appel à une fonction permettant d'associer chaque partie à une session.

In [14]:
from src.features.session_features import _add_sessions

df_player_games_v3 = _add_sessions(df_player_games_v2)

df_player_games_v3[["player_id", "datetime_utc", "delay_previous_game", "session_id"]].head(30)

,player_id,datetime_utc,delay_previous_game,session_id
149,chesstam,2026-03-06 21:09:47.356000+00:00,NaN,1
148,chesstam,2026-03-06 21:23:21.590000+00:00,814.234,1
147,chesstam,2026-03-06 21:29:47.688000+00:00,386.098,1
146,chesstam,2026-03-06 21:44:57.365000+00:00,909.677,1
145,chesstam,2026-03-06 21:58:16.384000+00:00,799.019,1
144,chesstam,2026-03-06 22:03:17.255000+00:00,300.871,1
143,chesstam,2026-03-07 05:33:55.132000+00:00,27037.877,2
142,chesstam,2026-03-07 05:40:42.844000+00:00,407.712,2
141,chesstam,2026-03-07 10:10:51.378000+00:00,16208.534,3
140,chesstam,2026-03-07 10:26:27.857000+00:00,936.479,3


L'expression ci-dessous réplique le bloc précédent et intègre des features associées aux sessions.

In [15]:
from src.features.session_features import add_sessions

df_player_games_v4 = add_sessions(df_player_games_v2)

df_player_games_v4[["player_id", "session_id", "session_position", "session_size", "session_score", "datetime_utc", "session_delay", "session_discrete_delay"]].head(30)

,player_id,session_id,session_position,session_size,session_score,datetime_utc,session_delay,session_discrete_delay
149,chesstam,1,0,6,0.833333,2026-03-06 21:09:47.356000+00:00,NaN,NaN
148,chesstam,1,1,6,0.833333,2026-03-06 21:23:21.590000+00:00,NaN,NaN
147,chesstam,1,2,6,0.833333,2026-03-06 21:29:47.688000+00:00,NaN,NaN
146,chesstam,1,3,6,0.833333,2026-03-06 21:44:57.365000+00:00,NaN,NaN
145,chesstam,1,4,6,0.833333,2026-03-06 21:58:16.384000+00:00,NaN,NaN
144,chesstam,1,5,6,0.833333,2026-03-06 22:03:17.255000+00:00,NaN,NaN
143,chesstam,2,0,2,0.000000,2026-03-07 05:33:55.132000+00:00,27037.877,B
142,chesstam,2,1,2,0.000000,2026-03-07 05:40:42.844000+00:00,27037.877,B
141,chesstam,3,0,2,0.000000,2026-03-07 10:10:51.378000+00:00,16208.534,A
140,chesstam,3,1,2,0.000000,2026-03-07 10:26:27.857000+00:00,16208.534,A


## 11.2 Série temporelle games_per_day

Même principe que pour la série sessions sauf qu'ici une session de jeu est une journée.

In [16]:
from src.features.day_features import add_day_features

df_player_games_v5 = add_day_features(df_player_games_v4)

df_player_games_v5[["player_id", "datetime_utc", "day_date_utc", "day_n_games", "day_score", "day_position"]].head(30)

,player_id,datetime_utc,day_date_utc,day_n_games,day_score,day_position
149,chesstam,2026-03-06 21:09:47.356000+00:00,2026-03-06,6,0.833333,0
148,chesstam,2026-03-06 21:23:21.590000+00:00,2026-03-06,6,0.833333,1
147,chesstam,2026-03-06 21:29:47.688000+00:00,2026-03-06,6,0.833333,2
146,chesstam,2026-03-06 21:44:57.365000+00:00,2026-03-06,6,0.833333,3
145,chesstam,2026-03-06 21:58:16.384000+00:00,2026-03-06,6,0.833333,4
144,chesstam,2026-03-06 22:03:17.255000+00:00,2026-03-06,6,0.833333,5
143,chesstam,2026-03-07 05:33:55.132000+00:00,2026-03-07,25,0.460000,0
142,chesstam,2026-03-07 05:40:42.844000+00:00,2026-03-07,25,0.460000,1
141,chesstam,2026-03-07 10:10:51.378000+00:00,2026-03-07,25,0.460000,2
140,chesstam,2026-03-07 10:26:27.857000+00:00,2026-03-07,25,0.460000,3


## 11.3 Série temporelle games_per_week

Même principe que pour la série sessions sauf qu'ici une session de jeu est une semaine.

In [17]:
from src.features.week_features import add_week_features

df_player_games_v6 = add_week_features(df_player_games_v5)

df_player_games_v6[["player_id", "datetime_utc", "week_id", "week_n_games", "week_score", "week_position", "termination_type", "result_player"]].head(30)

,player_id,datetime_utc,week_id,week_n_games,week_score,week_position,termination_type,result_player
149,chesstam,2026-03-06 21:09:47.356000+00:00,2026-W10,45,0.5,0,draw,0.5
148,chesstam,2026-03-06 21:23:21.590000+00:00,2026-W10,45,0.5,1,resign,1.0
147,chesstam,2026-03-06 21:29:47.688000+00:00,2026-W10,45,0.5,2,draw,0.5
146,chesstam,2026-03-06 21:44:57.365000+00:00,2026-W10,45,0.5,3,timeout,1.0
145,chesstam,2026-03-06 21:58:16.384000+00:00,2026-W10,45,0.5,4,resign,1.0
144,chesstam,2026-03-06 22:03:17.255000+00:00,2026-W10,45,0.5,5,mate,1.0
143,chesstam,2026-03-07 05:33:55.132000+00:00,2026-W10,45,0.5,6,mate,0.0
142,chesstam,2026-03-07 05:40:42.844000+00:00,2026-W10,45,0.5,7,resign,0.0
141,chesstam,2026-03-07 10:10:51.378000+00:00,2026-W10,45,0.5,8,mate,0.0
140,chesstam,2026-03-07 10:26:27.857000+00:00,2026-W10,45,0.5,9,resign,0.0


In [18]:
print(df_player_games_v6.columns)

Index(['game_id', 'datetime_utc', 'weekday', 'player_id', 'player_name',
       'opponent_id', 'opponent_name', 'color_player', 'result_player',
       'elo_player_before', 'elo_player_after', 'elo_diff_player',
       'termination_type', 'has_increment', 'opening_family', 'opening_eco',
       'opening_name', 'ply_count', 'perf', 'speed', 'rated', 'source',
       'index', 'delay_previous_game', 'streak_after', 'streak_before',
       'new_session', 'session_id', 'session_position', 'session_size',
       'session_score', 'is_session_start', 'session_delay',
       'session_discrete_delay', 'day_date_utc', 'day_n_games', 'day_score',
       'day_position', 'week_id', 'week_n_games', 'week_score',
       'week_position'],
      dtype='str')


# 12. Features joueur

## 12.1 Features liées au style de jeu

In [22]:
from src.features.player_features import _init_player_features
from src.features.player_features import _add_style_features

features = _init_player_features(df_player_games_v6) 
features_v1 = _add_style_features(df_player_games_v6, features)

features_v1[[
    "mean_ply_count",
    "main_opening_white",
    "main_opening_black",
    "opening_diversity",
    "opening_concentration"
]].head()

,mean_ply_count,main_opening_white,main_opening_black,opening_diversity,opening_concentration
player_id,,,,,
chesstam,65.5,Rapport-Jobava System,Scandinavian Defense,1.976381,0.386667


## 12.2 Features liées au comportement en fin de partie

In [23]:
from src.features.player_features import _add_endgame_behavior_features

features_v2 = _add_endgame_behavior_features(df_player_games_v6, features_v1)

features_v2[[
    "draw_ratio",
    "abandon_rate",
    "time_loss_rate"
]].head()

,draw_ratio,abandon_rate,time_loss_rate
player_id,,,
chesstam,0.033333,0.3,0.0
